<a href="https://www.kaggle.com/code/darahem/agent-ai-design-task?scriptVersionId=237300004" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install langgraph langchain openai
!pip install langchain_community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.2/151.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.2/437.2 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.35
    Uninstalling langchain-core-0.3.35:
      Successfully uninstalled langchain-core-0.3.35
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitt

In [2]:
"""
Meeting Room Booking Agent
--------------------------
A comprehensive implementation of an AI agent that helps users find and reserve meeting rooms,
built using LangGraph for workflow management and state transitions.
"""

from typing_extensions import TypedDict
from typing import List, Dict, Any, Optional, Tuple
from datetime import datetime, timedelta
import json
from langgraph.graph import StateGraph, END
from langchain.chat_models import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from pydantic import BaseModel, Field

# ===== State Management =====

class RoomFeature(BaseModel):
    """Model for room features with metadata"""
    name: str
    description: str = ""
    
class Room(BaseModel):
    """Model for meeting rooms"""
    id: str
    name: str
    features: List[str]
    capacity: int
    location: str
    floor: int

class TimeSlot(BaseModel):
    """Model for time slots"""
    start_time: datetime
    end_time: datetime

class Booking(BaseModel):
    """Model for room bookings"""
    id: str
    room_id: str
    user_id: str
    title: str
    description: str = ""
    start_time: datetime
    end_time: datetime
    attendees: List[str] = []

class UserPreference(BaseModel):
    """Model for tracking user preferences"""
    preferred_rooms: List[str] = []
    preferred_features: List[str] = []
    preferred_times: List[Tuple[int, int]] = []  # (hour, minute) tuples
    
class AgentState(TypedDict):
    """Complete state for the meeting room booking agent"""
    # Core booking information
    user_input: str
    conversation_history: List[Dict[str, Any]]
    
    # Intent and understanding
    intent: Optional[str]
    requested_date: Optional[datetime]
    requested_duration: Optional[int]  # minutes
    requested_time: Optional[datetime]
    requested_features: Optional[List[str]]
    requested_capacity: Optional[int]
    
    # Matching and selection
    matching_rooms: Optional[List[str]]
    room_suggestions: Optional[List[Dict[str, Any]]]
    selected_room: Optional[str]
    
    # Booking status
    reservation_status: Optional[str]
    booking_id: Optional[str]
    booking_conflicts: Optional[List[Dict[str, Any]]]
    
    # User information
    user_id: str
    user_preferences: Optional[UserPreference]
    
    # Error handling
    error: Optional[str]
    error_type: Optional[str]
    recovery_options: Optional[List[str]]
    
    # System state
    current_step: str
    previous_step: Optional[str]
    confirmation_required: bool
    
# ===== Mock External Systems =====

# Room database with more detailed information
ROOMS = {
    "rm-101": {
        "name": "Innovate", 
        "features": ["projector", "whiteboard", "video-conferencing"], 
        "capacity": 10,
        "location": "Building A",
        "floor": 1
    },
    "rm-102": {
        "name": "Collaborate", 
        "features": ["whiteboard", "tv-screen"], 
        "capacity": 5,
        "location": "Building A",
        "floor": 1
    },
    "rm-201": {
        "name": "Focus", 
        "features": ["tv-screen", "whiteboard", "quiet-room"], 
        "capacity": 8,
        "location": "Building A",
        "floor": 2
    },
    "rm-301": {
        "name": "Inspire", 
        "features": ["projector", "video-conferencing", "presentation-system"],
        "capacity": 20,
        "location": "Building B",
        "floor": 3
    },
}

# Mock calendar system
BOOKINGS = {}

# User profiles for preferences
USER_PROFILES = {
    "user-123": {
        "name": "Alex Johnson",
        "preferred_rooms": ["rm-101", "rm-301"],
        "preferred_features": ["whiteboard", "video-conferencing"],
        "preferred_times": [(9, 0), (14, 0)]  # 9 AM and 2 PM
    }
}

# ===== External System Integration Functions =====

def get_room_info(room_id: str) -> Dict[str, Any]:
    """Get room information from the database"""
    return ROOMS.get(room_id, {})

def get_all_rooms() -> Dict[str, Dict[str, Any]]:
    """Get all available rooms from the database"""
    return ROOMS

def check_availability(room_id: str, start_time: datetime, end_time: datetime) -> bool:
    """Check if a room is available for a given time slot"""
    for booking_id, booking in BOOKINGS.items():
        if booking['room_id'] == room_id:
            booking_start = booking['start_time']
            booking_end = booking['end_time']
            # Check for overlap
            if (start_time < booking_end and end_time > booking_start):
                return False
    return True

def create_booking(room_id: str, user_id: str, title: str, 
                  start_time: datetime, end_time: datetime,
                  description: str = "", attendees: List[str] = []) -> str:
    """Create a new booking and return the booking ID"""
    booking_id = f"bk-{len(BOOKINGS) + 1}"
    BOOKINGS[booking_id] = {
        "id": booking_id,
        "room_id": room_id,
        "user_id": user_id,
        "title": title,
        "description": description,
        "start_time": start_time,
        "end_time": end_time,
        "attendees": attendees
    }
    return booking_id

def cancel_booking(booking_id: str) -> bool:
    """Cancel an existing booking"""
    if booking_id in BOOKINGS:
        del BOOKINGS[booking_id]
        return True
    return False

def get_user_preferences(user_id: str) -> Dict[str, Any]:
    """Get user preferences from the user profile system"""
    return USER_PROFILES.get(user_id, {})

def get_suggested_rooms(features: List[str], capacity: int, 
                       start_time: datetime, end_time: datetime) -> List[Dict[str, Any]]:
    """Get suggested rooms based on features, capacity, and availability"""
    suggestions = []
    
    for room_id, room_info in ROOMS.items():
        if (capacity <= room_info['capacity'] and 
            all(feature in room_info['features'] for feature in features) and
            check_availability(room_id, start_time, end_time)):
            suggestions.append({
                "id": room_id,
                "name": room_info['name'],
                "features": room_info['features'],
                "capacity": room_info['capacity'],
                "location": room_info['location'],
                "floor": room_info['floor']
            })
    
    return suggestions

# ===== Agent Functions =====

def initialize_state(user_input: str, user_id: str = "user-123") -> AgentState:
    """Initialize the agent state with user input"""
    return {
        "user_input": user_input,
        "conversation_history": [{"role": "user", "content": user_input}],
        "intent": None,
        "requested_date": None,
        "requested_duration": None,
        "requested_time": None,
        "requested_features": None,
        "requested_capacity": None,
        "matching_rooms": None,
        "room_suggestions": None,
        "selected_room": None,
        "reservation_status": None,
        "booking_id": None,
        "booking_conflicts": None,
        "user_id": user_id,
        "user_preferences": None,
        "error": None,
        "error_type": None,
        "recovery_options": None,
        "current_step": "initialize",
        "previous_step": None,
        "confirmation_required": False
    }

def parse_intent(state: AgentState) -> AgentState:
    """Parse user input to determine intent"""
    user_input = state["user_input"].lower()
    
    # Update state tracking
    state["previous_step"] = state["current_step"]
    state["current_step"] = "parse_intent"
    
    # Simple intent classification
    if "book" in user_input or "reserve" in user_input or "schedule" in user_input:
        state["intent"] = "book_room"
    elif "cancel" in user_input:
        state["intent"] = "cancel_booking"
    elif "modify" in user_input or "change" in user_input or "reschedule" in user_input:
        state["intent"] = "modify_booking"
    elif "list" in user_input or "show" in user_input or "available" in user_input:
        state["intent"] = "list_rooms"
    elif "help" in user_input:
        state["intent"] = "help"
    else:
        state["intent"] = "unknown"
        
    return state

def extract_requirements(state: AgentState) -> AgentState:
    """Extract meeting requirements from user input"""
    user_input = state["user_input"].lower()
    
    # Update state tracking
    state["previous_step"] = state["current_step"]
    state["current_step"] = "extract_requirements"
    
    # Extract features
    features = []
    if "projector" in user_input:
        features.append("projector")
    if "whiteboard" in user_input:
        features.append("whiteboard")
    if "tv" in user_input or "screen" in user_input:
        features.append("tv-screen")
    if "video" in user_input or "conferenc" in user_input:
        features.append("video-conferencing")
    
    state["requested_features"] = features if features else None
    
    # Extract capacity
    capacity = None
    for word in user_input.split():
        if word.isdigit():
            num = int(word)
            if 1 <= num <= 50:  # Reasonable people count
                capacity = num
                break
    
    state["requested_capacity"] = capacity or 1  # Default to 1 person
    
    # Simple datetime extraction (in real implementation, use a proper NLP date parser)
    now = datetime.now()
    
    # Default to tomorrow at 2 PM if no specific time is mentioned
    tomorrow = now + timedelta(days=1)
    default_time = datetime(tomorrow.year, tomorrow.month, tomorrow.day, 14, 0)
    
    if "today" in user_input:
        meeting_date = now.date()
    elif "tomorrow" in user_input:
        meeting_date = tomorrow.date()
    else:
        meeting_date = tomorrow.date()  # Default to tomorrow
        
    state["requested_date"] = datetime.combine(meeting_date, datetime.min.time())
    
    # Extract time
    if "morning" in user_input:
        meeting_time = datetime.combine(meeting_date, datetime.min.time().replace(hour=10))
    elif "afternoon" in user_input:
        meeting_time = datetime.combine(meeting_date, datetime.min.time().replace(hour=14))
    elif "evening" in user_input:
        meeting_time = datetime.combine(meeting_date, datetime.min.time().replace(hour=17))
    else:
        meeting_time = datetime.combine(meeting_date, datetime.min.time().replace(hour=14))
    
    state["requested_time"] = meeting_time
    
    # Default duration: 1 hour
    state["requested_duration"] = 60
    
    return state

def load_user_preferences(state: AgentState) -> AgentState:
    """Load user preferences to influence room suggestions"""
    # Update state tracking
    state["previous_step"] = state["current_step"]
    state["current_step"] = "load_preferences"
    
    user_id = state["user_id"]
    preferences = get_user_preferences(user_id)
    
    if preferences:
        state["user_preferences"] = UserPreference(
            preferred_rooms=preferences.get('preferred_rooms', []),
            preferred_features=preferences.get('preferred_features', []),
            preferred_times=preferences.get('preferred_times', [])
        )
    
    return state

def find_matching_rooms(state: AgentState) -> AgentState:
    """Find rooms matching the requirements"""
    # Update state tracking
    state["previous_step"] = state["current_step"]
    state["current_step"] = "find_rooms"
    
    # Get requirements
    features = state["requested_features"] or []
    capacity = state["requested_capacity"] or 1
    
    # Calculate meeting end time
    start_time = state["requested_time"]
    duration = state["requested_duration"] or 60
    end_time = start_time + timedelta(minutes=duration)
    
    # Get room suggestions
    suggested_rooms = get_suggested_rooms(features, capacity, start_time, end_time)
    
    if suggested_rooms:
        # Sort by user preferences if available
        if state["user_preferences"] and state["user_preferences"].preferred_rooms:
            preferred_rooms = state["user_preferences"].preferred_rooms
            # Move preferred rooms to the top
            suggested_rooms.sort(key=lambda r: 0 if r["id"] in preferred_rooms else 1)
        
        state["room_suggestions"] = suggested_rooms
        state["matching_rooms"] = [room["id"] for room in suggested_rooms]
    else:
        state["error"] = "No available rooms match your criteria."
        state["error_type"] = "no_matching_rooms"
        state["recovery_options"] = ["adjust_time", "adjust_features", "adjust_capacity"]
    
    return state

def select_room(state: AgentState) -> AgentState:
    """Select the best room based on requirements and preferences"""
    # Update state tracking
    state["previous_step"] = state["current_step"]
    state["current_step"] = "select_room"
    
    if state["matching_rooms"]:
        # For now, just select the first room in the list
        # In a real system, this could use more sophisticated selection logic
        state["selected_room"] = state["matching_rooms"][0]
        
        # Get room details for the response
        room_info = get_room_info(state["selected_room"])
        
        # Add system message to conversation history
        system_msg = {
            "role": "system", 
            "content": f"Selected room: {room_info['name']} ({state['selected_room']})"
        }
        state["conversation_history"].append(system_msg)
        
        # Set confirmation required
        state["confirmation_required"] = True
    
    return state

def create_room_booking(state: AgentState) -> AgentState:
    """Create a booking for the selected room"""
    # Update state tracking
    state["previous_step"] = state["current_step"]
    state["current_step"] = "create_booking"
    
    if state["selected_room"]:
        room_id = state["selected_room"]
        user_id = state["user_id"]
        
        # Create meeting title from user input
        title = f"Meeting: {state['user_input'][:30]}"
        
        # Get timing information
        start_time = state["requested_time"]
        duration = state["requested_duration"]
        end_time = start_time + timedelta(minutes=duration)
        
        # Create the booking
        booking_id = create_booking(
            room_id=room_id,
            user_id=user_id,
            title=title,
            start_time=start_time,
            end_time=end_time
        )
        
        state["booking_id"] = booking_id
        state["reservation_status"] = "confirmed"
        
        # Room info for the response
        room_info = get_room_info(room_id)
        
        # Format time for display
        formatted_start = start_time.strftime("%A, %B %d at %I:%M %p")
        formatted_end = end_time.strftime("%I:%M %p")
        
        confirmation_msg = {
            "role": "system",
            "content": f"Booking confirmed! Room {room_info['name']} is reserved for {formatted_start} to {formatted_end}. Booking ID: {booking_id}"
        }
        state["conversation_history"].append(confirmation_msg)
    else:
        state["error"] = "Failed to create booking. No room selected."
        state["error_type"] = "booking_failed"
    
    return state

def handle_errors(state: AgentState) -> AgentState:
    """Handle errors and provide recovery options"""
    # Update state tracking
    state["previous_step"] = state["current_step"]
    state["current_step"] = "handle_errors"
    
    error = state.get("error")
    error_type = state.get("error_type")
    
    if error:
        error_msg = {
            "role": "system",
            "content": f"Error: {error}"
        }
        state["conversation_history"].append(error_msg)
        
        # Provide recovery options based on error type
        if error_type == "no_matching_rooms":
            recovery_msg = {
                "role": "system",
                "content": "Would you like to try different room features, a different time, or a smaller capacity?"
            }
            state["conversation_history"].append(recovery_msg)
        elif error_type == "booking_conflict":
            recovery_msg = {
                "role": "system",
                "content": "This room is already booked for the requested time. Would you like to see alternative times or different rooms?"
            }
            state["conversation_history"].append(recovery_msg)
    
    return state

def generate_response(state: AgentState) -> AgentState:
    """Generate a natural language response based on the current state"""
    # Update state tracking
    state["previous_step"] = state["current_step"]
    state["current_step"] = "generate_response"
    
    # Use LLM to generate a more natural response
    # In a real implementation, this would use an LLM
    # For now, we'll simulate with template responses
    
    intent = state.get("intent")
    
    if state.get("error"):
        # Error was already handled in handle_errors
        pass
    elif intent == "book_room" and state.get("reservation_status") == "confirmed":
        room_id = state.get("selected_room")
        room_info = get_room_info(room_id)
        
        response = {
            "role": "assistant",
            "content": f"Great! I've booked {room_info['name']} for you. It has "
                      f"{', '.join(room_info['features'])} and can accommodate {room_info['capacity']} people. "
                      f"Your booking ID is {state.get('booking_id')}. Anything else you need help with?"
        }
        state["conversation_history"].append(response)
    elif intent == "book_room" and state.get("room_suggestions"):
        suggestions = state.get("room_suggestions")
        if len(suggestions) > 1:
            response = {
                "role": "assistant",
                "content": f"I found {len(suggestions)} rooms that match your requirements. "
                          f"I recommend {suggestions[0]['name']} which has "
                          f"{', '.join(suggestions[0]['features'][:3])}. "
                          f"Would you like me to book this room for you?"
            }
        else:
            response = {
                "role": "assistant",
                "content": f"I found {suggestions[0]['name']} available for your meeting. "
                          f"It has {', '.join(suggestions[0]['features'])} and is located on floor {suggestions[0]['floor']}. "
                          f"Would you like me to book it for you?"
            }
        state["conversation_history"].append(response)
    elif intent == "list_rooms":
        if state.get("room_suggestions"):
            suggestions = state.get("room_suggestions")
            rooms_list = "\n".join([f"- {r['name']}: {', '.join(r['features'][:3])}" for r in suggestions[:3]])
            response = {
                "role": "assistant",
                "content": f"Here are some available rooms for your meeting:\n{rooms_list}\n\nWould you like to book any of these?"
            }
        else:
            response = {
                "role": "assistant",
                "content": "I couldn't find any rooms matching your criteria. Would you like to try different requirements?"
            }
        state["conversation_history"].append(response)
    elif intent == "unknown":
        response = {
            "role": "assistant",
            "content": "I'm not sure what you're looking for. I can help you book a meeting room, list available rooms, or cancel a booking. What would you like to do?"
        }
        state["conversation_history"].append(response)
    
    return state

# ===== Define the LangGraph workflow =====

def create_room_booking_agent():
    """Create the room booking agent workflow"""
    # Define the graph with state management
    workflow = StateGraph(AgentState)

    # Add nodes
    workflow.add_node("parse_intent", parse_intent)
    workflow.add_node("extract_requirements", extract_requirements)
    workflow.add_node("load_user_preferences", load_user_preferences)
    workflow.add_node("find_matching_rooms", find_matching_rooms)
    workflow.add_node("select_room", select_room)
    workflow.add_node("create_booking", create_room_booking)
    workflow.add_node("handle_errors", handle_errors)
    workflow.add_node("generate_response", generate_response)

    # Set up workflow
    workflow.set_entry_point("parse_intent")
    
    # Define edges
    workflow.add_edge("parse_intent", "extract_requirements")
    workflow.add_edge("extract_requirements", "load_user_preferences")
    workflow.add_edge("load_user_preferences", "find_matching_rooms")

    # Conditional edge: handle errors or continue based on room availability
    def should_continue_or_handle_error(state: AgentState) -> str:
        if state.get("error"):
            return "handle_errors"
        return "select_room"
    
    workflow.add_conditional_edges("find_matching_rooms", should_continue_or_handle_error, {
        "handle_errors": "handle_errors",
        "select_room": "select_room"
    })
    
    # Conditional edge: confirm booking or end flow
    def should_book_or_respond(state: AgentState) -> str:
        if state.get("intent") == "book_room" and state.get("selected_room") and not state.get("booking_id"):
            return "create_booking"
        return "generate_response"
    
    workflow.add_conditional_edges("select_room", should_book_or_respond, {
        "create_booking": "create_booking",
        "generate_response": "generate_response"
    })
    
    # Final edges
    workflow.add_edge("create_booking", "generate_response")
    workflow.add_edge("handle_errors", "generate_response")
    workflow.add_edge("generate_response", END)

    # Compile the graph
    return workflow.compile()

# ===== Testing the agent =====

def test_room_booking_agent():
    """Test the room booking agent with some example queries"""
    # Create the agent
    agent = create_room_booking_agent()
    
    # Test cases
    test_cases = [
        "I need to book a room with a projector for tomorrow afternoon",
        "Show me available rooms with video conferencing",
        "I want to schedule a meeting for 10 people with a whiteboard",
        "Book a room for tomorrow morning",
        "I need help using this system"
    ]
    
    results = []
    for i, query in enumerate(test_cases):
        print(f"\n===== Test Case {i+1}: {query} =====")
        initial_state = initialize_state(query)
        result = agent.invoke(initial_state)
        
        # Print conversation history
        for message in result["conversation_history"]:
            role = message["role"]
            content = message["content"]
            print(f"{role.capitalize()}: {content}")
        
        results.append(result)
    
    return results

if __name__ == "__main__":
    test_results = test_room_booking_agent()


===== Test Case 1: I need to book a room with a projector for tomorrow afternoon =====
User: I need to book a room with a projector for tomorrow afternoon
System: Selected room: Innovate (rm-101)
System: Booking confirmed! Room Innovate is reserved for Friday, May 02 at 02:00 PM to 03:00 PM. Booking ID: bk-1
Assistant: Great! I've booked Innovate for you. It has projector, whiteboard, video-conferencing and can accommodate 10 people. Your booking ID is bk-1. Anything else you need help with?

===== Test Case 2: Show me available rooms with video conferencing =====
User: Show me available rooms with video conferencing
System: Selected room: Inspire (rm-301)
Assistant: Here are some available rooms for your meeting:
- Inspire: projector, video-conferencing, presentation-system

Would you like to book any of these?

===== Test Case 3: I want to schedule a meeting for 10 people with a whiteboard =====
User: I want to schedule a meeting for 10 people with a whiteboard
System: Error: No ava